# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balabhadra3141/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Add your Hugging Face READ token as a Colab Secret named 'HF_TOKEN'."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Connected to FlyRank warehouse.")
print("Baseline decision window: February 2026")

Connected to FlyRank warehouse.
Baseline decision window: February 2026


## 1. My rule and its reason codes

### Rule

I will prioritize published content using two February signals:

1. **Search volume:** higher February impressions indicate greater observed search demand.
2. **Average position:** weaker February search visibility indicates a higher opportunity for review.

The score combines these two signals into one prioritization score. Higher scores receive higher priority.

### Reason codes

- `HIGH_VOLUME_LOW_VISIBILITY` — high search volume combined with weaker average position.
- `HIGH_VOLUME` — high search volume but position is not in the high-priority visibility band.
- `LOW_VISIBILITY` — weaker average position with lower search volume.
- `STANDARD_REVIEW` — does not meet the stronger conditions above.

### Action labels

- `PRIORITIZE_REVIEW` — review earlier.
- `REVIEW` — include in the review queue.
- `MONITOR` — lower-priority review.

This is a decision-support heuristic based only on February information. It is not a causal claim and does not use March outcomes or label-derived fields.

In [6]:
# Build the February decision-time feature frame.
# Only February information is used for the baseline.

baseline_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS impressions_feb,

        SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            ) > 0
            THEN
                CAST(
                    SUM(gsc_sum_position) FILTER (
                        WHERE gsc_data_available IS TRUE
                    ) AS DOUBLE
                )
                /
                SUM(gsc_impressions) FILTER (
                    WHERE gsc_data_available IS TRUE
                )
            ELSE NULL
        END AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.avg_position_feb

FROM feb f

JOIN '{DIM_CONTENT}' c
    ON f.content_hash_id = c.content_hash_id

WHERE
    f.impressions_feb >= 100
    AND f.clicks_feb >= 3
    AND f.avg_position_feb IS NOT NULL
    AND c.is_published IS TRUE
""").df()

print("Baseline rows:", len(baseline_df))
display(baseline_df.head())

Baseline rows: 29700


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb
0,client_3ffa76342f366962,content_32bdebcb01540202,551.0,17.0,3.758621
1,client_e547b89c05043229,content_4c1e972bec56132e,2882.0,15.0,10.403539
2,client_e547b89c05043229,content_4e48bd81bb37eb4f,7343.0,3.0,45.407054
3,client_e547b89c05043229,content_7e131483384291cf,6206.0,6.0,8.934096
4,client_e547b89c05043229,content_3f12146e57e4baf0,16877.0,29.0,4.443444


## 2. Build the ranked queue

The score gives more weight to pages with stronger observed search demand and weaker search visibility.

I use percentile-based thresholds rather than arbitrary raw thresholds so the rule reflects the observed February distribution.

The score is:

- +2 for high search volume (at or above the 75th percentile)
- +2 for weaker visibility (average position above the 75th percentile)
- +1 for moderately high search volume (at or above the median)
- +1 for moderately weak visibility (average position above the median)

The resulting score ranges from 0 to 4.

The queue is ranked by score, then by February impressions and average position.

In [7]:
import pandas as pd
import numpy as np

# Copy the February-only feature frame.
queue = baseline_df.copy()

# Calculate thresholds from the February decision-time data.
volume_median = queue["impressions_feb"].median()
volume_q75 = queue["impressions_feb"].quantile(0.75)

position_median = queue["avg_position_feb"].median()
position_q75 = queue["avg_position_feb"].quantile(0.75)

print("February thresholds")
print(f"Volume median:      {volume_median:.2f}")
print(f"Volume 75th pct:    {volume_q75:.2f}")
print(f"Position median:    {position_median:.2f}")
print(f"Position 75th pct:  {position_q75:.2f}")

February thresholds
Volume median:      2561.00
Volume 75th pct:    5370.00
Position median:    5.42
Position 75th pct:  8.74


In [15]:
# Score the two confirmed signals.
# Higher impressions = stronger observed demand.
# Higher average position = weaker observed visibility.

queue["score"] = (
    np.where(queue["impressions_feb"] >= volume_q75, 2, 0)
    + np.where(
        (queue["impressions_feb"] >= volume_median)
        & (queue["impressions_feb"] < volume_q75),
        1,
        0
    )
    + np.where(queue["avg_position_feb"] >= position_q75, 2, 0)
    + np.where(
        (queue["avg_position_feb"] >= position_median)
        & (queue["avg_position_feb"] < position_q75),
        1,
        0
    )
)

# One reason code per row.
queue["reason_code"] = np.select(
    [
        (queue["impressions_feb"] >= volume_q75)
        & (queue["avg_position_feb"] >= position_q75),

        (queue["impressions_feb"] >= volume_q75),

        (queue["avg_position_feb"] >= position_q75),
    ],
    [
        "HIGH_VOLUME_LOW_VISIBILITY",
        "HIGH_VOLUME",
        "LOW_VISIBILITY",
    ],
    default="STANDARD_REVIEW"
)

# Action label.
queue["action"] = np.select(
    [
        queue["score"] >= 4,
        queue["score"] >= 2,
    ],
    [
        "PRIORITIZE_REVIEW",
        "REVIEW",
    ],
    default="MONITOR"
)

# Rank highest score first.
queue = queue.sort_values(
    by=[
        "score",
        "impressions_feb",
        "avg_position_feb"
    ],
    ascending=[
        False,
        False,
        False
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions_feb",
            "clicks_feb",
            "avg_position_feb",
            "score",
            "reason_code",
            "action",
        ]
    ].head(20)
)

,rank,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,score,reason_code,action
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,627.0,13.584411,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
1,2,client_23a62021009f63c4,content_36e53e9c707674fc,100736.0,237.0,34.405029,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
2,3,client_23a62021009f63c4,content_df47d1b976106de4,85935.0,103.0,17.186094,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
3,4,client_fef1a8f436438636,content_84a6bf3578312e90,79986.0,74.0,19.783062,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,73072.0,118.0,16.379119,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
5,6,client_fef1a8f436438636,content_ba462518dad435fc,69134.0,39.0,27.477479,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
6,7,client_23a62021009f63c4,content_3df3f32f3fd58dea,68216.0,155.0,24.829497,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
7,8,client_23a62021009f63c4,content_b51957d7f4abe47e,57558.0,36.0,28.411064,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
8,9,client_23a62021009f63c4,content_bdf60c86117079be,51346.0,8.0,33.837300,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW
9,10,client_fef1a8f436438636,content_0aaa197051f58d6f,49903.0,35.0,34.489710,4,HIGH_VOLUME_LOW_VISIBILITY,PRIORITIZE_REVIEW


In [16]:
# sanity checks
print("Rows in ranked queue:", len(queue))
print("Score range:", queue["score"].min(), "to", queue["score"].max())

print("\nAction counts:")
display(queue["action"].value_counts())

print("\nReason-code counts:")
display(queue["reason_code"].value_counts())

Rows in ranked queue: 29700
Score range: 0 to 4

Action counts:


,count
action,
MONITOR,15015
REVIEW,12960
PRIORITIZE_REVIEW,1725



Reason-code counts:


,count
reason_code,
STANDARD_REVIEW,16573
HIGH_VOLUME,5702
LOW_VISIBILITY,5700
HIGH_VOLUME_LOW_VISIBILITY,1725


In [17]:
# Write the required CSV to "work/outputs/baseline_action_score.csv"
from pathlib import Path

OUTPUT_PATH = Path("work/outputs/baseline_action_score.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
    "score",
    "reason_code",
    "action",
]

queue[output_columns].to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Wrote {len(queue):,} rows to {OUTPUT_PATH}")

Wrote 29,700 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

The top-20 queue is reviewed manually rather than treated as automatically correct.

For each row, I record the action, the reason code, a confidence note, and what could make the recommendation wrong. The confidence note reflects how directly the row matches the rule, not the probability that the action will succeed.

In [18]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["score"] == 4:
        return "Strong rule match: both confirmed signals are high-priority."
    elif row["score"] == 3:
        return "Moderate-to-strong rule match: one signal is high-priority and the other is moderate."
    elif row["score"] == 2:
        return "Moderate rule match: one confirmed signal reaches the high-priority threshold."
    else:
        return "Lower-confidence rule match."

def what_would_make_it_wrong(row):
    if row["reason_code"] == "HIGH_VOLUME_LOW_VISIBILITY":
        return "The February signals may not reflect the actual content opportunity or search intent."
    elif row["reason_code"] == "HIGH_VOLUME":
        return "High impressions may not represent a useful optimization opportunity."
    elif row["reason_code"] == "LOW_VISIBILITY":
        return "Lower visibility may be caused by factors the two-feature rule cannot observe."
    else:
        return "The available February signals may be insufficient to justify the review."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "score",
            "confidence_note",
            "what_would_make_it_wrong",
        ]
    ]
)

,rank,content_hash_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_e8a52cf3d5988c07,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
1,2,content_36e53e9c707674fc,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
2,3,content_df47d1b976106de4,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
3,4,content_84a6bf3578312e90,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
4,5,content_5e1c049f62e33b11,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
5,6,content_ba462518dad435fc,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
6,7,content_3df3f32f3fd58dea,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
7,8,content_b51957d7f4abe47e,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
8,9,content_bdf60c86117079be,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...
9,10,content_0aaa197051f58d6f,PRIORITIZE_REVIEW,HIGH_VOLUME_LOW_VISIBILITY,4,Strong rule match: both confirmed signals are ...,The February signals may not reflect the actua...


## 4. Weak picks + leakage check

Some recommendations can look weak even when the rule is applied correctly. In particular, a high score does not prove that a page should actually be changed.

Possible weak picks include pages where high search volume does not translate into a meaningful optimization opportunity, or where average position is affected by query mix or other factors not represented in this baseline.

The baseline uses February impressions, clicks, and average position plus publication status. It does not use March outcomes, `trend_direction`, product flags, or future-window measurements.

In [19]:
# inspect weak/ambiguous picks
weak_picks = queue[
    (queue["score"] <= 1)
].head(10)

print("Example lower-scoring picks:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions_feb",
            "clicks_feb",
            "avg_position_feb",
            "score",
            "reason_code",
            "action",
        ]
    ]
)

Example lower-scoring picks:


,rank,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,score,reason_code,action
14685,14686,content_c33a3cfc9364d685,5369.0,22.0,5.126094,1,STANDARD_REVIEW,MONITOR
14686,14687,content_8ef0c9f3c7071a35,5369.0,36.0,3.042839,1,STANDARD_REVIEW,MONITOR
14687,14688,content_712c365258cee05c,5368.0,21.0,3.818927,1,STANDARD_REVIEW,MONITOR
14688,14689,content_bbd8ed0b66ba914c,5367.0,4.0,4.103969,1,STANDARD_REVIEW,MONITOR
14689,14690,content_2c95bca7710be3eb,5366.0,10.0,2.388185,1,STANDARD_REVIEW,MONITOR
14690,14691,content_6e12e9a3f72aebc1,5366.0,30.0,1.082930,1,STANDARD_REVIEW,MONITOR
14691,14692,content_3a5a681e2b575fd4,5365.0,3.0,3.170923,1,STANDARD_REVIEW,MONITOR
14692,14693,content_56e58d1dc9f4d5c2,5365.0,9.0,2.841193,1,STANDARD_REVIEW,MONITOR
14693,14694,content_498481c22c79b90b,5364.0,12.0,5.121738,1,STANDARD_REVIEW,MONITOR
14694,14695,content_6d22c35c85342d15,5363.0,24.0,3.178445,1,STANDARD_REVIEW,MONITOR


In [20]:
# Leakage check
# Explicitly check that no known future/label-derived fields are present in the baseline queue.

forbidden_columns = [
    "trend_direction",
    "march_clicks",
    "march_impressions",
    "march_zero_click_rate",
    "label",
    "target",
    "future_clicks",
    "future_impressions",
]

found_forbidden = [
    col for col in forbidden_columns
    if col in queue.columns
]

print("Forbidden future/label-derived columns found:", found_forbidden)

assert not found_forbidden, (
    f"Leakage check failed. Found: {found_forbidden}"
)

print("Leakage check: PASS")

Forbidden future/label-derived columns found: []
Leakage check: PASS


In [21]:
# Additional source-level check
# The baseline feature columns are explicitly limited to February decision-time measurements.

baseline_feature_columns = [
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
]

print("Baseline feature columns:")
for col in baseline_feature_columns:
    print("-", col)

print("\nNo March outcome columns are used in scoring.")

Baseline feature columns:
- impressions_feb
- clicks_feb
- avg_position_feb

No March outcome columns are used in scoring.


## Self-check

- [x] Every section is filled — markdown reasoning and supporting code are present.
- [x] The notebook runs top to bottom with no errors.
- [x] The rule uses only February decision-time information.
- [x] The baseline uses the two signals supported by the ML-06 tests: search volume and average position.
- [x] Content age was not forced into the rule because its observed relationship was mixed.
- [x] `days_since_last_update` is excluded because its February values were not a clean pre-decision measure.
- [x] The queue contains a score, one reason code, and an action label.
- [x] `work/outputs/baseline_action_score.csv` is generated by the notebook.
- [x] The top 20 rows have an action, reason code, confidence note, and what would make the recommendation wrong.
- [x] No future-window outcomes, product flags, or label-derived fields are used in the baseline score.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language: observed, measured, directional, and decision-support.
- [x] The notebook is committed under `work/notebooks/w04_baseline_score.ipynb`.